# MMLU-Pro result analysis

This notebook analyzes all `*_result.json` files for one evaluated model.

Place this notebook in `eval_results/` and set `MODEL_DIR` below, for example:

```text
eval_results/
├── mmlu_pro_analysis.ipynb
├── llama3_8b/
│   ├── business_result.json
│   ├── biology_result.json
│   └── ...
└── qwen_14b/
    └── ...
```

The notebook derives metrics directly from the result JSON files rather than trusting the generated summary files.

In [ ]:
from pathlib import Path
import json
import math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Notebook is expected to live in eval_results/.
MODEL_DIR = Path("llama3_8b")

# MMLU-Pro uses answer labels A-J.
LABELS = list("ABCDEFGHIJ")

# Save generated figures next to the selected model results.
SAVE_FIGURES = False
FIGURE_DIR = MODEL_DIR / "analysis_figures"

if SAVE_FIGURES:
  FIGURE_DIR.mkdir(parents=True, exist_ok=True)

MODEL_DIR.resolve()

## 1. Load all present categories

Every `*_result.json` file is treated as one category. The category is read from the result data when available; otherwise it is inferred from the filename.

In [ ]:
result_files = sorted(MODEL_DIR.glob("*_result.json"))

if not result_files:
  raise FileNotFoundError(
    f"No *_result.json files found in {MODEL_DIR.resolve()}"
  )

frames = []

for path in result_files:
  with path.open("r", encoding="utf-8") as f:
    rows = json.load(f)

  if not isinstance(rows, list):
    raise ValueError(f"{path} does not contain a JSON list")

  df = pd.DataFrame(rows)
  if df.empty:
    continue

  if "category" not in df.columns:
    df["category"] = path.stem.removesuffix("_result")

  df["source_file"] = path.name
  frames.append(df)

results = pd.concat(frames, ignore_index=True)

required = {"question_id", "category", "answer", "pred"}
missing = required - set(results.columns)
if missing:
  raise ValueError(f"Missing required columns: {sorted(missing)}")

results["answer"] = results["answer"].astype("string").str.upper()
results["pred"] = results["pred"].astype("string").str.upper()
results["correct"] = results["answer"] == results["pred"]

categories = sorted(results["category"].dropna().unique())

print(f"Model directory : {MODEL_DIR}")
print(f"Result files    : {len(result_files)}")
print(f"Categories      : {len(categories)}")
print(f"Questions       : {len(results)}")
print(f"Correct         : {int(results['correct'].sum())}")
print(f"Overall accuracy: {results['correct'].mean():.4%}")

## 2. Sanity checks

These checks are useful before interpreting benchmark accuracy. In particular:

- missing predictions can make the score look artificially low,
- invalid labels indicate answer-extraction problems,
- duplicate question IDs may indicate interrupted or merged runs,
- a heavily skewed prediction distribution may reveal positional bias.

In [ ]:
valid_labels = set(LABELS)

null_pred = results["pred"].isna().sum()
invalid_pred = (~results["pred"].isin(LABELS) & results["pred"].notna()).sum()
invalid_answer = (~results["answer"].isin(LABELS) & results["answer"].notna()).sum()

duplicate_ids = results.duplicated(
  subset=["category", "question_id"],
  keep=False,
).sum()

sanity = pd.DataFrame({
  "check": [
    "Questions",
    "Missing predictions",
    "Invalid prediction labels",
    "Invalid answer labels",
    "Duplicate category/question IDs",
  ],
  "value": [
    len(results),
    int(null_pred),
    int(invalid_pred),
    int(invalid_answer),
    int(duplicate_ids),
  ],
})

sanity

## 3. Accuracy and bias summary by category

`max_pred_share` is the fraction of predictions occupied by the most frequently predicted answer label. A value near `0.10` would be close to uniform across ten labels; a much larger value is worth investigating.

`prediction_entropy` is normalized to the range 0–1. Lower values indicate a more concentrated answer distribution.

In [ ]:
def normalized_entropy(series, labels=LABELS):
  counts = series.value_counts().reindex(labels, fill_value=0).to_numpy(dtype=float)
  total = counts.sum()
  if total == 0:
    return np.nan

  p = counts / total
  p = p[p > 0]
  entropy = -(p * np.log(p)).sum()
  return entropy / np.log(len(labels))


summary_rows = []

for category, df in results.groupby("category", sort=True):
  pred_counts = df["pred"].value_counts().reindex(LABELS, fill_value=0)
  top_label = pred_counts.idxmax()
  top_count = int(pred_counts.max())

  summary_rows.append({
    "category": category,
    "questions": len(df),
    "correct": int(df["correct"].sum()),
    "wrong": int((~df["correct"]).sum()),
    "accuracy": df["correct"].mean(),
    "missing_pred": int(df["pred"].isna().sum()),
    "top_pred_label": top_label,
    "top_pred_count": top_count,
    "max_pred_share": top_count / len(df),
    "prediction_entropy": normalized_entropy(df["pred"]),
  })

category_summary = pd.DataFrame(summary_rows).sort_values(
  "accuracy",
  ascending=False,
).reset_index(drop=True)

category_summary

In [ ]:
fig, ax = plt.subplots(figsize=(10, max(4, 0.45 * len(category_summary))))

plot_df = category_summary.sort_values("accuracy")
ax.barh(plot_df["category"], plot_df["accuracy"] * 100)
ax.set_xlabel("Accuracy (%)")
ax.set_ylabel("Category")
ax.set_title("MMLU-Pro accuracy by category")
ax.grid(axis="x", alpha=0.25)

plt.tight_layout()

if SAVE_FIGURES:
  fig.savefig(FIGURE_DIR / "accuracy_by_category.png", dpi=160, bbox_inches="tight")

plt.show()

## 4. Correct-answer vs prediction distribution

This is the quickest way to detect positional bias. The benchmark's correct answers should normally be much more balanced than a strongly biased model prediction distribution.

In [ ]:
answer_counts = results["answer"].value_counts().reindex(LABELS, fill_value=0)
pred_counts = results["pred"].value_counts().reindex(LABELS, fill_value=0)

distribution = pd.DataFrame({
  "answer_count": answer_counts,
  "prediction_count": pred_counts,
})

distribution["answer_share"] = distribution["answer_count"] / distribution["answer_count"].sum()
distribution["prediction_share"] = distribution["prediction_count"] / distribution["prediction_count"].sum()
distribution.index.name = "label"

distribution

In [ ]:
x = np.arange(len(LABELS))
width = 0.4

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x - width / 2, distribution["answer_share"] * 100, width, label="Correct answer")
ax.bar(x + width / 2, distribution["prediction_share"] * 100, width, label="Model prediction")

ax.set_xticks(x)
ax.set_xticklabels(LABELS)
ax.set_xlabel("Answer label")
ax.set_ylabel("Share (%)")
ax.set_title("Correct-answer vs prediction distribution")
ax.legend()
ax.grid(axis="y", alpha=0.25)

plt.tight_layout()

if SAVE_FIGURES:
  fig.savefig(FIGURE_DIR / "answer_vs_prediction_distribution.png", dpi=160, bbox_inches="tight")

plt.show()

## 5. Per-label precision and recall

For a benchmark answer label:

- **Recall**: when that label is actually correct, how often did the model choose it?
- **Precision**: when the model chose that label, how often was it correct?

A label with high recall but low precision is typically being over-predicted.

In [ ]:
label_metrics = []

for label in LABELS:
  actual = results["answer"] == label
  predicted = results["pred"] == label
  tp = (actual & predicted).sum()

  recall = tp / actual.sum() if actual.sum() else np.nan
  precision = tp / predicted.sum() if predicted.sum() else np.nan

  label_metrics.append({
    "label": label,
    "actual_count": int(actual.sum()),
    "predicted_count": int(predicted.sum()),
    "true_positive": int(tp),
    "precision": precision,
    "recall": recall,
  })

label_metrics = pd.DataFrame(label_metrics).set_index("label")
label_metrics

In [ ]:
x = np.arange(len(LABELS))
width = 0.4

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x - width / 2, label_metrics["precision"] * 100, width, label="Precision")
ax.bar(x + width / 2, label_metrics["recall"] * 100, width, label="Recall")

ax.set_xticks(x)
ax.set_xticklabels(LABELS)
ax.set_xlabel("Answer label")
ax.set_ylabel("Percent")
ax.set_title("Per-label precision and recall")
ax.legend()
ax.grid(axis="y", alpha=0.25)

plt.tight_layout()

if SAVE_FIGURES:
  fig.savefig(FIGURE_DIR / "per_label_precision_recall.png", dpi=160, bbox_inches="tight")

plt.show()

## 6. Confusion matrices for every present category

Each matrix is **row-normalized**:

- rows = correct answer,
- columns = model prediction,
- values = percentage of questions with that correct answer.

This makes positional bias easier to compare between categories of different sizes.

In [ ]:
def confusion_matrix_percent(df, labels=LABELS):
  matrix = pd.crosstab(
    df["answer"],
    df["pred"],
    dropna=False,
  ).reindex(index=labels, columns=labels, fill_value=0)

  row_sums = matrix.sum(axis=1).replace(0, np.nan)
  normalized = matrix.div(row_sums, axis=0) * 100

  return matrix, normalized


n = len(categories)
cols = min(3, n)
rows = math.ceil(n / cols)

fig, axes = plt.subplots(
  rows,
  cols,
  figsize=(5.2 * cols, 4.8 * rows),
  squeeze=False,
)

for ax, category in zip(axes.flat, categories):
  df = results[results["category"] == category]
  _, matrix_pct = confusion_matrix_percent(df)

  image = ax.imshow(matrix_pct.to_numpy(), vmin=0, vmax=100, aspect="auto")

  ax.set_title(f"{category} (n={len(df)})")
  ax.set_xlabel("Predicted")
  ax.set_ylabel("Correct")
  ax.set_xticks(np.arange(len(LABELS)))
  ax.set_yticks(np.arange(len(LABELS)))
  ax.set_xticklabels(LABELS)
  ax.set_yticklabels(LABELS)

  for i in range(len(LABELS)):
    for j in range(len(LABELS)):
      value = matrix_pct.iloc[i, j]
      if pd.notna(value) and value >= 1:
        ax.text(
          j,
          i,
          f"{value:.0f}",
          ha="center",
          va="center",
          fontsize=7,
        )

for ax in axes.flat[n:]:
  ax.axis("off")

fig.suptitle("MMLU-Pro confusion matrices by category — row-normalized (%)", y=1.002)
fig.colorbar(image, ax=axes.ravel().tolist(), shrink=0.65, label="Percent")
fig.subplots_adjust(hspace=0.35, wspace=0.3)

if SAVE_FIGURES:
  fig.savefig(
    FIGURE_DIR / "confusion_matrices_all_categories.png",
    dpi=160,
    bbox_inches="tight",
  )

plt.show()

## 7. Raw confusion matrix for a selected category

Useful when investigating one suspicious category in detail.

In [ ]:
SELECTED_CATEGORY = "business"

if SELECTED_CATEGORY not in categories:
  raise ValueError(
    f"{SELECTED_CATEGORY!r} is not present. Available: {categories}"
  )

selected = results[results["category"] == SELECTED_CATEGORY]
matrix_raw, matrix_pct = confusion_matrix_percent(selected)

matrix_raw

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))
image = ax.imshow(matrix_raw.to_numpy(), aspect="auto")

ax.set_title(f"{SELECTED_CATEGORY}: raw confusion matrix")
ax.set_xlabel("Predicted")
ax.set_ylabel("Correct")
ax.set_xticks(np.arange(len(LABELS)))
ax.set_yticks(np.arange(len(LABELS)))
ax.set_xticklabels(LABELS)
ax.set_yticklabels(LABELS)

for i in range(len(LABELS)):
  for j in range(len(LABELS)):
    value = int(matrix_raw.iloc[i, j])
    if value:
      ax.text(
        j,
        i,
        str(value),
        ha="center",
        va="center",
        fontsize=9,
      )

fig.colorbar(image, ax=ax, label="Questions")
plt.tight_layout()

if SAVE_FIGURES:
  fig.savefig(
    FIGURE_DIR / f"{SELECTED_CATEGORY}_confusion_matrix_raw.png",
    dpi=160,
    bbox_inches="tight",
  )

plt.show()

## 8. Prediction distribution by category

This table makes answer-position bias visible across categories. Values are percentages of all predictions in that category.

In [ ]:
prediction_share_by_category = (
  pd.crosstab(results["category"], results["pred"], normalize="index")
  .reindex(columns=LABELS, fill_value=0)
  * 100
)

prediction_share_by_category.round(2)

In [ ]:
fig, ax = plt.subplots(
  figsize=(12, max(4, 0.45 * len(prediction_share_by_category)))
)

image = ax.imshow(prediction_share_by_category.to_numpy(), aspect="auto")

ax.set_title("Prediction-label share by category (%)")
ax.set_xlabel("Predicted label")
ax.set_ylabel("Category")
ax.set_xticks(np.arange(len(LABELS)))
ax.set_xticklabels(LABELS)
ax.set_yticks(np.arange(len(prediction_share_by_category)))
ax.set_yticklabels(prediction_share_by_category.index)

for i in range(len(prediction_share_by_category)):
  for j in range(len(LABELS)):
    value = prediction_share_by_category.iloc[i, j]
    if value >= 1:
      ax.text(
        j,
        i,
        f"{value:.0f}",
        ha="center",
        va="center",
        fontsize=7,
      )

fig.colorbar(image, ax=ax, label="Prediction share (%)")
plt.tight_layout()

if SAVE_FIGURES:
  fig.savefig(
    FIGURE_DIR / "prediction_share_by_category.png",
    dpi=160,
    bbox_inches="tight",
  )

plt.show()

## 9. Most suspicious category/label combinations

This ranks category/label combinations by how much the model's prediction share exceeds the true-answer share.

A large positive `share_delta` means that answer position is being selected much more frequently than it occurs as the correct answer.

In [ ]:
bias_rows = []

for category, df in results.groupby("category", sort=True):
  answer_share = (
    df["answer"].value_counts(normalize=True)
    .reindex(LABELS, fill_value=0)
  )
  pred_share = (
    df["pred"].value_counts(normalize=True)
    .reindex(LABELS, fill_value=0)
  )

  for label in LABELS:
    bias_rows.append({
      "category": category,
      "label": label,
      "answer_share": answer_share[label],
      "prediction_share": pred_share[label],
      "share_delta": pred_share[label] - answer_share[label],
    })

bias_table = pd.DataFrame(bias_rows).sort_values(
  "share_delta",
  ascending=False,
)

bias_table.head(30)

## 10. Inspect individual questions

Use this to inspect the strongest bias directly. Change `FILTER_CATEGORY`, `FILTER_PRED`, or `ONLY_WRONG` as needed.

In [ ]:
FILTER_CATEGORY = "business"
FILTER_PRED = "H"
ONLY_WRONG = True

inspect_df = results.copy()

if FILTER_CATEGORY:
  inspect_df = inspect_df[inspect_df["category"] == FILTER_CATEGORY]

if FILTER_PRED:
  inspect_df = inspect_df[inspect_df["pred"] == FILTER_PRED]

if ONLY_WRONG:
  inspect_df = inspect_df[~inspect_df["correct"]]

columns = [
  col for col in [
    "question_id",
    "category",
    "question",
    "options",
    "answer",
    "pred",
    "model_outputs",
  ]
  if col in inspect_df.columns
]

inspect_df[columns].head(50)

## 11. Export compact analysis tables

Optional CSV exports are useful for the thesis and for comparing multiple quantizations later.

In [ ]:
EXPORT_TABLES = False

if EXPORT_TABLES:
  export_dir = MODEL_DIR / "analysis_tables"
  export_dir.mkdir(parents=True, exist_ok=True)

  category_summary.to_csv(
    export_dir / "category_summary.csv",
    index=False,
  )
  distribution.to_csv(
    export_dir / "answer_prediction_distribution.csv",
  )
  label_metrics.to_csv(
    export_dir / "label_metrics.csv",
  )
  prediction_share_by_category.to_csv(
    export_dir / "prediction_share_by_category.csv",
  )
  bias_table.to_csv(
    export_dir / "position_bias.csv",
    index=False,
  )

  print(f"Exported tables to {export_dir.resolve()}")

## Interpretation checklist

Before treating a category score as a model-quality result, check:

1. `Missing predictions == 0`.
2. Prediction labels are valid A-J.
3. Correct-answer distribution is reasonably balanced.
4. Prediction distribution is not dominated by one position without explanation.
5. Confusion matrices do not show an obvious systematic extraction or positional artifact.
6. If comparing against the official leaderboard, document differences in:
   - exact model checkpoint,
   - quantization,
   - inference backend,
   - chat template,
   - few-shot/CoT prompt,
   - temperature/top-p,
   - max output tokens,
   - stopping criteria,
   - answer extraction/scoring behavior.
